# 第2课：从单个神经元到多神经元层

**学习目标：**
- 理解单个神经元的计算过程（加权和 + 偏置）
- 理解 ReLU 激活函数的作用
- 用矩阵运算实现多个神经元
- 理解批处理（batch）的含义和广播机制

---

神经网络的基本单元是**神经元**。一个神经元接收多个输入，乘以各自的权重，加上偏置，再通过激活函数输出结果。本课从最简单的单神经元开始，逐步过渡到矩阵化的多神经元计算。

## 2.1 单个神经元（感知器）

最简单的神经元计算：$output = \sum(x_i \cdot w_i) + b$

例如 3 个输入的神经元：

In [ ]:
import numpy as np

# 输入
a1, a2, a3 = 0.9, 0.5, 0.7

# 权重
w1, w2, w3 = 0.8, -0.4, 0.0

# 偏置
b = 1.0

# 加权和
output = a1*w1 + a2*w2 + a3*w3 + b
print("加权和:", output)  # 1.52

## 2.2 ReLU 激活函数

**为什么需要激活函数？** 如果没有非线性激活，多层网络等价于单层线性变换，无法学习复杂模式。

ReLU（Rectified Linear Unit）是最常用的激活函数：

$$ReLU(x) = \max(0, x)$$

- 正数保持不变
- 负数变为 0

In [ ]:
def activation_ReLU(x):
    return np.maximum(0, x)

print("ReLU(1.52):", activation_ReLU(output))     # 1.52（正数不变）
print("ReLU(-1.52):", activation_ReLU(-output))   # 0.0（负数变0）

## 2.3 向量化：用矩阵运算替代循环

用 `np.array` 和 `np.dot` 重写神经元计算，代码更简洁、运行更快：

$$output = X \cdot W + b$$

其中 $X$ 是输入向量，$W$ 是权重向量。

In [ ]:
inputs = np.array([0.9, 0.5, 0.7])    # shape: (3,)
weights = np.array([0.8, -0.4, 0.0])  # shape: (3,)
bias = 1.0

# np.dot 计算向量点积 = 加权和
output = np.dot(inputs, weights) + bias
print("向量化计算结果:", output)  # 1.52，与逐元素计算结果相同

## 2.4 多个神经元：权重矩阵

当我们有多个神经元时，每个神经元都有自己的权重和偏置。将它们组合成**矩阵**，一次计算就能得到所有神经元的输出：

$$Z = X \cdot W + b$$

- $X$: shape `(1, n)` — 1 个样本，n 个特征
- $W$: shape `(n, p)` — p 个神经元，每个有 n 个权重
- $b$: shape `(p,)` — p 个偏置
- $Z$: shape `(1, p)` — p 个神经元的输出

In [ ]:
# 3个输入，2个神经元
inputs = np.array([0.9, 0.5, 0.7])  # shape: (3,)

# 权重矩阵：每列对应一个神经元的权重
weights = np.array([[0.8, 0.7],   # 神经元1的w1, 神经元2的w1
                    [-0.4, -0.6], # 神经元1的w2, 神经元2的w2
                    [0.0, 0.2]])  # 神经元1的w3, 神经元2的w3
# shape: (3, 2)

bias = np.array([0.5, 0.5])  # 每个神经元一个偏置

# 矩阵乘法：一次计算所有神经元
z = np.dot(inputs, weights) + bias  # (3,) x (3,2) + (2,) = (2,)
print("加权和:", z)
print("ReLU 激活后:", activation_ReLU(z))

## 2.5 批处理（Batch）

实际训练时，我们会同时处理多个样本。将多个样本堆成矩阵，一次前向传播就能处理整个 batch：

$$Z = X \cdot W + b$$

- $X$: shape `(m, n)` — m 个样本，每个 n 维
- $W$: shape `(n, p)` — p 个神经元
- $b$: shape `(p,)` — 通过**广播**自动扩展到每个样本
- $Z$: shape `(m, p)` — 每个样本对应 p 个输出

In [ ]:
# 3个样本，每个3维特征
inputs = np.array([[0.9, 0.5, 0.7],
                    [0.8, 0.5, 0.6],
                    [0.5, 0.8, 0.2]])  # shape: (3, 3)

# 权重矩阵不变
weights = np.array([[0.8, 0.7],
                    [-0.4, -0.6],
                    [0.0, 0.2]])  # shape: (3, 2)

bias = np.array([0.5, 0.5])  # shape: (2,)，自动广播到3个样本

# 一次计算：3个样本 × 2个神经元
z = np.dot(inputs, weights) + bias
print("加权和 (3个样本, 2个神经元):")
print(z)
print("\nReLU 激活后:")
print(activation_ReLU(z))

## 2.6 自动生成权重和偏置

实际使用中，权重和偏置用随机数初始化。`np.random.randn` 生成标准正态分布的随机数。

In [ ]:
def create_weights(input_size, output_size):
    """创建权重矩阵：(输入维度, 神经元数)"""
    return np.random.randn(input_size, output_size)

def create_biases(n_neurons):
    """创建偏置向量：(神经元数,)"""
    return np.random.randn(n_neurons)

# 示例：3个输入 → 2个神经元
W = create_weights(3, 2)
b = create_biases(2)
print("权重矩阵 W (3×2):")
print(W)
print("\n偏置 b:", b)

---

## 维度速查表

| 符号 | 含义 | 数学维度 | 说明 |
| --- | --- | --- | --- |
| $X$ | 输入 | $(m, n)$ | m 个样本，每个 n 维特征 |
| $W$ | 权重 | $(n, p)$ | p 个神经元，每个连 n 个输入 |
| $b$ | 偏置 | $(p,)$ | 每个神经元一个，广播到 m 个样本 |
| $Z$ | 输出 | $(m, p)$ | m 个样本，每个 p 维输出 |

---

## 小结

- 单个神经元：$output = \sum(x_i \cdot w_i) + b$
- ReLU：$\max(0, x)$，引入非线性
- 多神经元 = 矩阵乘法：$Z = X \cdot W + b$
- 批处理：多行输入矩阵，一次算出所有结果
- 广播：偏置 $(p,)$ 自动扩展到 $(m, p)$

**下一课**我们将把这些计算封装成 `Layer` 和 `Network` 类。